In [1]:
import xarray as xr
import earthaccess
import boto3
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import warnings
from IPython.display import display, Markdown
import pandas as pd
import geopandas as gpd
import rasterio
import datetime
import pyarrow as pa
import pyarrow.parquet as pq
import os
import numpy as np

warnings.filterwarnings('ignore')
%matplotlib inline

In [ ]:
fires = pd.read_parquet("s3://maap-ops-workspace/shared/zbecker/TESS_fire_spread/sigdeltas_Tess.parq")
subset_fires = gpd.read_parquet("s3://maap-ops-workspace/shared/zbecker/YANG/large_feds_faf_double_matched.parq")
subset_fires = subset_fires.to_crs(4326)

subset_fires["centroid"] = subset_fires.to_crs(4326).centroid
fires["UfireID"] = fires.mergeid.astype("int").astype("str") + "_" + fires.year.astype("str")
subset_fires["UfireID"] = subset_fires.mergeid.astype("str") + "_" + subset_fires.year.astype("str")
subset_fires["polygon"] = subset_fires.geometry
fires = fires[fires.UfireID.isin(subset_fires[subset_fires.intersectsMTBS == True].UfireID)]
fname = pd.read_csv("s3://maap-ops-workspace/shared/zbecker/Eli_MTBS_vs_FEDS/v6_output.csv")

fires = fires.merge(subset_fires[['UfireID', 'centroid', 'polygon']], on = 'UfireID' )
fires = gpd.GeoDataFrame(fires, geometry = 'polygon')

def get_st_sp_fire(df, days_after = 7):
    df.loc[:, "start_time"] = df.t.min()
    df.loc[:, "end_time"] = df.t.max()
    df.loc[:, "end_time_plus"] = df.t.astype("datetime64[ns]").max()  + datetime.timedelta(days = days_after)
    df = df.loc[df.t == df.t.max(), :]
    return(df)
    
fires_sm = fires.groupby("UfireID").apply(get_st_sp_fire).reset_index(drop = True)
fires_sm["stable_index"] = fires_sm.index

In [ ]:
precip = pd.read_parquet(os.path.abspath("IMERG/half_hourly_IMERG_precip"))

In [ ]:
precip = precip.merge(fires_sm[["UfireID",	"centroid"]], on = "UfireID")

precip.loc[:, "lon"] = precip.centroid.apply(lambda p: p.x)

In [ ]:
precip.loc[:, "offset_hour"] = (precip.lon/15)

precip.loc[:, "time_lst"] = precip.time_utc.astype("datetime64[ns]") + pd.to_timedelta(precip["offset_hour"], unit='h') #lst_to_utc_offset_hours =  lon/ 15.0

pm_mask = (precip.time_lst.dt.hour > 6) & (precip.time_lst.dt.hour <= 18) ## correcting to PM 13:30 overpass
am_mask = (precip.time_lst.dt.hour <= 6) | ((precip.time_lst.dt.hour >= 18))## correcting to AM 1:30 overpass. This is actually an exact  number not a range bc we calcuated it for the extraction. 

precip.loc[pm_mask, "t"] = precip.loc[pm_mask, "time_lst"].dt.normalize() + pd.Timedelta(hours=12)
precip.loc[am_mask, "t"] = precip.loc[am_mask, "time_lst"].dt.normalize() + pd.Timedelta(hours=0)





In [ ]:
## Finding days_since_t
fires.t = fires.t.astype('datetime64[ns]')
fires = fires.merge(precip[[ 'precipitation', 'UfireID', 't']], how = 'outer', on = ['UfireID', 't'])

fires = fires.merge(fires_sm[['UfireID', 'start_time', 'end_time','end_time_plus']], on = 'UfireID')

# foo = fires[fires.UfireID == "10000_2019"]
# foo = foo.sort_values(by = "t")
# foo




## Extraction included the precip values from BEFORE the fire's first growth increment

In [ ]:
## Calculating time offset from ending
fires['end_time_offset'] =  fires.t.astype("datetime64[ns]") - fires.end_time.astype("datetime64[ns]") # Positive means days past the end date
fires['start_time_offset'] = fires.t.astype("datetime64[ns]") - fires.start_time.astype("datetime64[ns]") # Positive means days past the start date, negative means before
fires['start_off_12hrs'] = fires['start_time_offset'] / pd.Timedelta(hours=12) 
fires['end_off_12hrs'] = fires['end_time_offset'] / pd.Timedelta(hours=12)

### calculating timeoffset from begining

# Plots!

In [ ]:

af_mask =  (fires['start_off_12hrs'] >= 0) & (fires['end_off_12hrs'] <= 0)

plt.scatter(y = fires[af_mask].area_growth_at_t_km2, x = fires[af_mask].precipitation)
plt.xlabel("precipitation (mm/12 hr) @ t")
plt.ylabel("area growth km^2 @ t")
plt.title("Precipitation During Active Fire Growth")
plt.suptitle(f"{fires[af_mask].UfireID.nunique()} fires, from {fires[af_mask].t.dt.year.min()} to {fires[af_mask].t.dt.year.max()}")

# Zeb's dataset doesn't have the non-spread timesteps, so we don't have the times when firegrowth = 0 in the above plot

In [ ]:
fires.loc[af_mask & (fires.area_growth_at_t_km2.isna()), "area_growth_at_t_km2"] = 0

In [ ]:
plt.scatter(y = fires[af_mask].area_growth_at_t_km2, x = fires[af_mask].precipitation)
plt.xlabel("precipitation (mm/12 hr) @ t")
plt.ylabel("area growth km^2 @ t")
plt.title("Precipitation from ft0-ftfinal (aka now we have where 0 fire growth)")
plt.suptitle(f"{fires[af_mask].UfireID.nunique()} fires, from {fires[af_mask].t.dt.year.min()} to {fires[af_mask].t.dt.year.max()}")

fires.loc[:, "precipitation_no_zero"] = fires["precipitation"] + 1
fires.loc[:, "area_growth_at_t_km2_no_zero"] = fires["area_growth_at_t_km2"] + 1

In [ ]:
import seaborn as sns

sns.jointplot(data = fires[af_mask], y = "area_growth_at_t_km2", x = "precipitation")
plt.xscale('log')
plt.yscale('log')

In [ ]:
sns.regplot(y = np.log(fires[af_mask].area_growth_at_t_km2_no_zero), x = np.log(fires[af_mask].precipitation_no_zero), ci = None, line_kws=dict(color="r"))

In [ ]:
sns.regplot(y = np.log(fires.loc[af_mask & (fires.area_growth_at_t_km2 > 0)  & (fires.precipitation > 0),: ].area_growth_at_t_km2), x = np.log(fires.loc[af_mask & (fires.area_growth_at_t_km2 > 0)  & (fires.precipitation > 0),: ].precipitation), ci = 95, line_kws=dict(color="r"))

#### How many fires ended after precipitation? 

I will look at the distribution of precip 
 - after fires ended (12 hour increment to start out)
 - during fires
 - before fires started (12 hour increment)
 - if ending precip bigger than during precip important? (dunno)





In [ ]:
import seaborn as sns

### set up column for easly plotting 

pre_fire_mask = fires.start_off_12hrs == -1 ## conservative, next 12 hours, could expand to 24 because of differences in detection effecientcy
post_fire_mask = fires.end_off_12hrs == 1


fires.loc[pre_fire_mask, "fr_active_status"] = "pre_fire"
fires.loc[post_fire_mask, "fr_active_status"] = "post_fire"
fires.loc[af_mask, "fr_active_status"] = "during_fire"


sns.violinplot(data=fires[["fr_active_status", "precipitation_no_zero"]].dropna(), x = "fr_active_status" , y="precipitation_no_zero", log_scale=(False, True))

plt.title("Precip by fire status")
plt.show()

In [ ]:
sns.violinplot(data=fires[["fr_active_status", "precipitation"]].dropna(), x = "fr_active_status" , y="precipitation")

plt.title("Precip by fire status")
plt.xlabel("12 hours increments")
plt.ylabel("precipitation mm/12hr")
plt.show()

Hard to see from the above plots, but I think that postfire has a higher central tendency than the others. Maybe look at the distributions CI's and use a KSTEST? 



In [ ]:
print("70th Quantile for different fire statuses")

print(fires.groupby("fr_active_status").precipitation.quantile(0.7))


print("80th Quantile for different fire statuses")

print(fires.groupby("fr_active_status").precipitation.quantile(0.8))

print("90th Quantile for different fire statuses")

print(fires.groupby("fr_active_status").precipitation.quantile(0.9))



### TO DO KSTEST

# How many fires ended after an *uptick* in precipitation? 

Instead of looking at the raw amount of precipitation after a fire stopped, looking at fires that stopped after an uptick in precip specifically. This is a little scary given the 12-hourly stuff, so may need to consider a rolling difference, or 24 hour difference etc. 

In [ ]:
fires = fires.merge(subset_fires[['n_pixels', 'n_newpixels',
       'farea', 'duration', 'pixden', 'meanFRP',  'UfireID']], on = 'UfireID')

gsp = gpd.read_file("/home/jovyan/Preparedness_level/GACC_borders/National_GACC_Boundaries.shp")

In [ ]:
#fires = fires.sjoin(gsp[['GACCName', 'GACCAbbrev','geometry']])

In [ ]:
fires.t = fires.t.astype("datetime64[ns]")
fires = fires.sort_values(by = ["UfireID", "t"])
fires["precip_diff"] = fires.groupby("UfireID").precipitation.diff() ## Positive is when the amount of rain goes up, negative down

In [ ]:
fires.loc[post_fire_mask,:].precip_diff.hist(bins = 25) ## For the 12 hours after the last spread, seems like a a mean 1.58 mm per 12 hours
plt.xlabel( " t_1 - t_0 difference in precip (mm/12 hour)")
plt.ylabel( "count")
print("Difference in precip was t_1 - t_0")
print(f" There was a mean of {fires.loc[post_fire_mask,:].precip_diff.mean()} difference in precip in the 12 hours after last fire growth")

print(f" 50th quantile {fires.loc[post_fire_mask,:].precip_diff.quantile(.5)}")
print(f" 60th quantile {fires.loc[post_fire_mask,:].precip_diff.quantile(.6)}")
print(f" 70th quantile {fires.loc[post_fire_mask,:].precip_diff.quantile(.7)}")
print(f" 80th quantile {fires.loc[post_fire_mask,:].precip_diff.quantile(.8)}")
print(f" 90th quantile {fires.loc[post_fire_mask,:].precip_diff.quantile(.9)}")

fires.loc[post_fire_mask & (fires.precip_diff > 0),"precip_end_event"] = True
fires.loc[post_fire_mask & (fires.precip_diff <= 0),"precip_end_event"] = False

print(f"{len(fires.loc[post_fire_mask & (fires.precip_diff > 0),:].precip_diff) / len(fires.loc[post_fire_mask ,:].precip_diff)} Percent of fires ended after a positve increase in precip in the 12 hours after the last fire growth detection.")

In [ ]:
# Of the positives, what was the range in precipitation values? 

fires.loc[post_fire_mask & (fires.precip_diff > 0),:].precip_diff.hist() # I would love to match this to final fire size. I bet there is is trade off!

print(fires.loc[post_fire_mask & (fires.precip_diff > 0),:].precip_diff.min())

In [ ]:
gsp = gsp.to_crs("EPSG:4326")
gsp = gsp.dissolve(by='GACCAbbrev').reset_index()

fires_gacc = fires.sjoin(gsp[['GACCName', 'GACCAbbrev','geometry']])

In [ ]:
## Percent of fires where weather stayed the same


print(f"{len(fires.loc[post_fire_mask & (fires.precip_diff == 0),:].precip_diff) / len(fires.loc[post_fire_mask ,:].precip_diff)} Percent of fires ended after no change in precip in the 12 hours after the last fire growth detection.")

In [ ]:
print(f"{len(fires.loc[post_fire_mask & (fires.precip_diff < 0),:].precip_diff) / len(fires.loc[post_fire_mask ,:].precip_diff)} Percent of fires ended after a decrease in precip in the 12 hours after the last fire growth detection.")


print(f" NOTE Percentages coming from {len(fires.loc[post_fire_mask ,:].UfireID.unique())} fires.")

#### Check for uncertainty in the % numbers, pull out by catagory. 

In [ ]:
n_iterations = 10000
overall_bootstraps = np.empty(n_iterations)


target_array =fires.loc[post_fire_mask, 'precip_end_event'].values 

for i in range(n_iterations):
    # Resample with replacement
    sample = np.random.choice(target_array, size=len(target_array), replace=True)
    overall_bootstraps[i] = np.mean(sample)

overall_obs = np.mean(target_array)
overall_ci = np.percentile(overall_bootstraps, [2.5, 97.5])

print("--- OVERALL UNCERTAINTY ---")
print(f"Observed Overall Proportion: {overall_obs:.1%}")
print(f"95% Confidence Interval:     [{overall_ci[0]:.1%}, {overall_ci[1]:.1%}]\n")


# Abstracting to 

Looking at the percentage (and confidence intervals) on a catagorical basis. ecoregions and GACCS

In [ ]:
# # ==========================================
# # 2. Subset Sensitivity (Difference Bootstrap)
# # ==========================================
# group_a = df[df['is_freshwater'] == True]['is_blue'].values
# group_b = df[df['is_freshwater'] == False]['is_blue'].values

# diff_bootstraps = np.empty(n_iterations)

# for i in range(n_iterations):
#     # Bootstrap within each group independently
#     sample_a = np.random.choice(group_a, size=len(group_a), replace=True)
#     sample_b = np.random.choice(group_b, size=len(group_b), replace=True)
    
#     # Calculate the difference in proportions
#     diff_bootstraps[i] = np.mean(sample_a) - np.mean(sample_b)

# obs_diff = np.mean(group_a) - np.mean(group_b)
# diff_ci = np.percentile(diff_bootstraps, [2.5, 97.5])

# print("--- SUBSET SENSITIVITY (Freshwater vs Non-Freshwater) ---")
# print(f"Observed Freshwater Prop:     {np.mean(group_a):.1%}")
# print(f"Observed Non-Freshwater Prop: {np.mean(group_b):.1%}")
# print(f"Observed Difference:          {obs_diff*100:.1f} percentage points")
# print(f"95% CI of Difference:         [{diff_ci[0]*100:.1f}, {diff_ci[1]*100:.1f}] percentage points")

# if diff_ci[0] > 0 or diff_ci[1] < 0:
#     print("\nCONCLUSION: The CI does not cross 0. The subset is SIGNIFICANTLY different.")
# else:
#     print("\nCONCLUSION: The CI crosses 0. The subset is NOT significantly different.")

# # Optional: Plotting the difference distribution
# plt.hist(diff_bootstraps, bins=50, color='skyblue', edgecolor='black')
# plt.axvline(0, color='red', linestyle='--', label='Zero Difference (No Effect)')
# plt.axvline(diff_ci[0], color='green', linestyle=':', label='95% CI Bounds')
# plt.axvline(diff_ci[1], color='green', linestyle=':')
# plt.title('Bootstrap Distribution of Difference in Proportions')
# plt.xlabel('Difference in Proportion (Group A - Group B)')
# plt.ylabel('Frequency')
# plt.legend()
# plt.show()

In [ ]:
### Check if this works beter for multi-day fires

In [ ]:
# Fix columsn taht should be constans on a per-id basis

def per_id_constants(df, cols = ["year", "l1_ecoregion", 	"centroid"]):
    for col in cols:
        col_val = df.loc[~df[col].isna(), col].iloc[0]
        df.loc[:, col] = col_val
    return(df)
fires = fires.groupby("UfireID").apply(per_id_constants).reset_index(drop = True)

# Look at fire-stopping precip by GACCS and by ecoregion

In [ ]:

just_post_and_during_mask = fires.fr_active_status.isin(['during_fire', 'post_fire'])

#
sns.violinplot(data=fires.loc[just_post_and_during_mask, ["fr_active_status", 'precipitation_no_zero', "l1_ecoregion"]].dropna(), x = "l1_ecoregion" , y='precipitation_no_zero', hue = "fr_active_status", inner = None, log_scale = [False, True])

plt.title("Precip After Growth vs During growth")
plt.xlabel("12 hours increments")
plt.ylabel("precipitation mm/12hr")
plt.xticks(rotation=90)
plt.show()


just_post_and_during_mask = fires.fr_active_status.isin(['during_fire', 'post_fire'])

#
sns.violinplot(data=fires.loc[:, ["fr_active_status", 'precipitation_no_zero', "l1_ecoregion"]].dropna(), x = "l1_ecoregion" , y='precipitation_no_zero', hue = "fr_active_status", log_scale = [False, True], inner = None)

plt.title("Precip by fire status")
plt.xlabel("12 hours increments")
plt.ylabel("precipitation mm/12hr")
plt.xticks(rotation=90)
plt.show()

In [ ]:
### Why i smatplotlib so weird about color mapping?

# levels, categories = pd.factorize(fires[af_mask].fr_active_status)
# colors = [plt.cm.tab10(i) for i in levels] # using the "tab10" colormap

sns.scatterplot(data = fires[af_mask], y = 'area_growth_at_t_km2', x = "precipitation", hue = "l1_ecoregion" )
#plt.scatter(y = fires[af_mask].area_growth_at_t_km2, x = fires[af_mask].precipitation, c = colors)
plt.xlabel("precipitation (mm/12 hr) @ t")
plt.ylabel("area growth km^2 @ t")
plt.title("Precipitation from ft0-ftfinal (aka now we have where 0 fire growth)")
plt.suptitle(f"{fires[af_mask].UfireID.nunique()} fires, from {fires[af_mask].t.dt.year.min()} to {fires[af_mask].t.dt.year.max()}")
plt.show()

In [ ]:

cat_var = "l1_ecoregion"
for ec in fires.loc[af_mask, cat_var].unique():
    sns.lmplot(data = fires.loc[af_mask & (fires[cat_var] == ec), :], x = "precipitation_no_zero", y = 'area_growth_at_t_km2', 
           lowess = True, ci=None,  line_kws=dict(color="r"))
    # plt.xlabel("precipitation (mm/12 hr) @ t")
    plt.ylabel("area growth km^2 @ t")
    plt.title(f"{ec}")
    plt.suptitle(f"{fires.loc[af_mask & (fires[cat_var] == ec), :].UfireID.nunique()} fires, from {fires.loc[af_mask & (fires[cat_var] == ec), :].t.dt.year.min()} to {fires.loc[af_mask & (fires[cat_var] == ec), :].t.dt.year.max()}")
    plt.show()
    
# sns.lmplot(data = fires.loc[af_mask & (fires[cat_var] == ec), :], x = "precipitation", y = 'area_growth_at_t_km2', 
#        order=2, ci=None, col = "l1_ecoregion")

# Look at active fire (smoldering) time vs actualy expansion time